<a href="https://colab.research.google.com/github/Aireenelz/WIE3007-DMW-Group6/blob/xgboost-modelling-joan/xgboost_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# 1. Setup and Data Loading

import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv('synthetic_financial_data.csv')

# Quick look
print(f"Dataset Shape: {df.shape}")
df.head()

Dataset Shape: (1000, 22)


,CustomerID,Age,Education,EmploymentType,EmploymentSector,EmploymentLengthYears,MonthlyIncome,MaritalStatus,Dependents,PropertyOwnership,...,MonthlyDebt,YearsWithBank,HasSavingsAccount,HasCheckingAccount,LoanPurpose,LoanAmount,LoanTermMonths,InterestRate,LoanDefault,LoanPurposeDescription
0,1,59,High School,Self-Employed,Technology,11,9067.159786,Single,2,Own with Mortgage,...,1583.065500,7,1,0,Personal,20789.005502,36,3.828919,0,Need funds for the next three years.
1,2,51,Below High School,Full-Time,Government,24,5068.058742,Married,1,Own with Mortgage,...,1332.052087,7,1,1,Personal,27461.583712,48,8.855769,1,I need a personal loan for this to be done.
2,3,24,Bachelor,Unemployed,Manufacturing,0,2557.541824,Single,0,Living with Parents,...,911.536097,2,1,1,Business,68862.744645,240,14.748993,1,Seeking financing to start my own company.
3,4,25,Bachelor,Full-Time,Finance,0,7535.097299,Single,0,Living with Parents,...,514.141526,6,1,0,Wedding,8841.450954,48,4.825886,0,I need a personal loan for me.
4,5,25,Master,Part-Time,Government,5,4798.465251,Married,2,Rent,...,2386.488285,3,0,1,Education,17787.927594,120,18.384157,1,"Applying for student financing for tuition, fe..."


#### 2. Small Language Models (SLM) Feature Extraction
##### - use DistillBERT model for sentiment analysis of column 'LoanPurposeDescription'
##### - create a new column 'Sentiment_Feature' that categorize the 'LoanPurposeDescription' into 'POSITIVE', 'NEGATIVE, and 'NEUTRAL'


In [14]:
# import necessary library
from transformers import pipeline

# Load a lightweight sentiment model (distillbert model) for sentiment analysis
sentiment_task = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
# sentiment-analysis: tell the model we want sentiment analysis so it looks for positive or negative emotions in text

# get_sentiment: function to process text
def get_sentiment(text):
    # if a row has no text, just mark it as "NEUTRAL"
    if pd.isna(text) or text == "": return "NEUTRAL"
    # [:512]: the limit because BER-style model can only read 512 characters at a time
    # ['label']: The model returns a lot of data, but we only want the 'label' (e.g., "POSITIVE" or "NEGATIVE").
    return sentiment_task(text[:512])[0]['label']

# create a new column 'Sentiment_Feature'
# it takes the text in LoanPurposeDescription and converts it into a category
df['Original_Sentiment_Feature'] = df['LoanPurposeDescription'].apply(get_sentiment)
print(df['Original_Sentiment_Feature'].value_counts())
df[['LoanPurposeDescription', 'Sentiment_Feature']].head(10)

Device set to use cuda:0


Original_Sentiment_Feature
NEGATIVE    872
NEUTRAL      66
POSITIVE     62
Name: count, dtype: int64


,LoanPurposeDescription,Sentiment_Feature
0,Need funds for the next three years.,NEGATIVE
1,I need a personal loan for this to be done.,NEGATIVE
2,Seeking financing to start my own company.,NEGATIVE
3,I need a personal loan for me.,NEGATIVE
4,"Applying for student financing for tuition, fe...",POSITIVE
5,Applying for financing to cover the cost of th...,NEGATIVE
6,I need a personal loan for me.,NEGATIVE
7,Applying for financing to cover the costs of t...,NEGATIVE
8,Applying for financing to cover the costs of t...,NEGATIVE
9,I need a personal loan for me to take care of ...,NEGATIVE


##### It is very common for financial datasets to lean "negative" when using general sentiment models because words like "Debt," "Emergency," or "Loan" are often flagged as negative.

To get a better mix of Positive, Negative, and Neutral, a Keyword-Augmented Logic is used.

The Strategy:
- Neutral: Keep NaN as Neutral.

- Positive Keywords: If words like "Growth," "Expansion," "Investment," or "Success" appear, we nudge the result to Positive.

- Negative Keywords: If words like "Struggling," "Debt," "Emergency," or "Late" appear, we nudge it to Negative.

- AI Fallback: If no keywords are found, we let the DistilBERT model decide.

In [6]:
# improve the sentiment analysis

# import necessary library
from transformers import pipeline

# Load a lightweight sentiment model (distillbert model) for sentiment analysis
sentiment_task = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
# sentiment-analysis: tell the model we want sentiment analysis so it looks for positive or negative emotions in text

# get_sentiment: function to process text
def get_improved_sentiment(text):
    # # if a row has no text, just mark it as "NEUTRAL"
    if pd.isna(text) or text == "" or text == "None":
        return "NEUTRAL"

    text_lower = str(text).lower()

    # define Financial Keywords
    positive_words = ['business', 'expand', 'opportunity','student', 'starting', 'future','career','education','new']
    negative_words = ['struggling', 'debt', 'emergency', 'overdue', 'bills', 'medical', 'crisis', 'urgent']

    # keyword
    if any(word in text_lower for word in positive_words):
        return "POSITIVE"
    if any(word in text_lower for word in negative_words):
        return "NEGATIVE"

    # AI Fallback (If no keywords found, let the model decide)
    try:
        result = sentiment_task(text[:512])[0]['label']
        return result
    except:
        return "NEUTRAL"

# Apply the function
df['Sentiment_Feature'] = df['LoanPurposeDescription'].apply(get_improved_sentiment)

print(df['Sentiment_Feature'].value_counts())
df[['LoanPurposeDescription', 'Sentiment_Feature']].head(10)

Device set to use cuda:0


Sentiment_Feature
NEGATIVE    634
POSITIVE    300
NEUTRAL      66
Name: count, dtype: int64


,LoanPurposeDescription,Sentiment_Feature
0,Need funds for the next three years.,NEGATIVE
1,I need a personal loan for this to be done.,NEGATIVE
2,Seeking financing to start my own company.,NEGATIVE
3,I need a personal loan for me.,NEGATIVE
4,"Applying for student financing for tuition, fe...",POSITIVE
5,Applying for financing to cover the cost of th...,NEGATIVE
6,I need a personal loan for me.,NEGATIVE
7,Applying for financing to cover the costs of t...,NEGATIVE
8,Applying for financing to cover the costs of t...,NEGATIVE
9,I need a personal loan for me to take care of ...,NEGATIVE
